# Очистка данных

Ноутбук выполняет очистку объединённого датасета:

1. Удаление пустых и коротких текстов (< 5 символов)
2. Фильтрация текстов без буквенно-цифровых слов
3. Точная дедупликация по `text`
4. Near-дедупликация (нормализованная)
5. Near-дедупликация (MinHash LSH, Jaccard >= 0.8)
6. Удаление пересечений с тестовой выборкой (data leakage)

Загружается `data/interim/raw_combined.csv`, результат сохраняется в `data/interim/cleaned.csv`.

In [1]:
import os
import re
import sys
import numpy as np
import pandas as pd

try:
    _project_root = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
except NameError:
    _cwd = os.getcwd()
    _project_root = _cwd
    while _project_root != '/' and not os.path.isdir(os.path.join(_project_root, 'src')):
        _project_root = os.path.dirname(_project_root)
if _project_root not in sys.path:
    sys.path.insert(0, _project_root)

from src.data.loaders import load_txt_lines
from src.config import RAW_DIR, EXTERNAL_DIR, INTERIM_DIR

import emoji
import regex as regex_lib
from datasketch import MinHash, MinHashLSH

In [2]:
df = pd.read_csv(INTERIM_DIR / 'raw_combined.csv')
print(f'Загружено: {len(df)} строк')

Загружено: 566420 строк


### Очистка данных

Этапы очистки:
1. Удаление пустых и слишком коротких текстов (< 5 символов)
2. Фильтрация текстов без буквенно-цифровых слов
3. Точная дедупликация по полю `text`
4. Near-дедупликация по нормализованному тексту (lowercase + удаление пунктуации + нормализация пробелов + удаление эмодзи)
5. Near-дедупликация через MinHash LSH (Jaccard similarity > 0.8)
6. Удаление текстов с конфликтными метками (один текст — разные классы)
7. Удаление пересечений с тестовой выборкой (data leakage)

#### Удаление пустых и слишком коротких текстов

Удаляются строки с пустым текстом или длиной менее 5 символов.
Также удаляются тексты, не содержащие ни одного буквенно-цифрового слова длиной >= 2
(сообщения, состоящие только из эмодзи, ссылок или пунктуации).

In [3]:
df_2 = df[df['text'].notna()]
df_2 = df_2[df_2['text'].str.strip() != '']
df_2 = df_2[df_2['text'].str.len() >= 5]
# Фильтрация текстов без буквенно-цифровых слов длиной >= 2
_has_word = df_2['text'].str.contains(r'\b[a-zA-Zа-яА-ЯёЁ0-9]{2,}\b', regex=True, na=False)
df_2 = df_2[_has_word]

In [4]:
print(f'После удаления пустых/коротких (<5 символов) и без слов: {len(df_2)} строк')

После удаления пустых/коротких (<5 символов) и без слов: 537198 строк


#### Точная дедупликация

Удаляются полные дубликаты по полю `text` (оставляется первое вхождение).
Конфликты меток уже проверены до объединения.

In [5]:
before_dedup = len(df_2)
df_3 = df_2.drop_duplicates(subset='text', keep='first').copy()

In [6]:
print(f'Удалено дубликатов: {before_dedup - len(df_3)}')
print(f'После дедупликации: {len(df_3)} строк')

Удалено дубликатов: 427349
После дедупликации: 109849 строк


#### Near-дедупликация (нормализованная)

Поиск near-дубликатов после нормализации текста:
- приведение к нижнему регистру
- удаление пунктуации
- нормализация пробелов
- удаление эмодзи

Это позволяет выявить сообщения, отличающиеся только регистром,
пунктуацией или набором эмодзи.

In [7]:
def _normalize_for_dedup(text: str) -> str:
    """Нормализация текста для поиска near-дубликатов."""
    text = text.lower().strip()
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^\w\s]', '', text)
    # Удаление эмодзи через grapheme clusters
    import emoji as _emoji
    import regex as _regex
    _graphemes = _regex.findall(r'\X', text)
    text = ''.join(g for g in _graphemes if not _emoji.is_emoji(g))
    return text.strip()

In [8]:
_before_near = len(df_3)
df_3['_norm'] = df_3['text'].apply(_normalize_for_dedup)
df_4 = df_3.drop_duplicates(subset='_norm', keep='first').drop(columns=['_norm'])
_near_removed = _before_near - len(df_4)
print(f'Удалено near-дубликатов (нормализованных): {_near_removed}')
print(f'После near-дедупликации: {len(df_4)} строк')

Удалено near-дубликатов (нормализованных): 3259
После near-дедупликации: 106590 строк


#### Near-дедупликация (MinHash LSH)

Поиск нечётких дубликатов через MinHash LSH (Jaccard similarity > 0.8).
Используется библиотека `datasketch` для эффективного поиска
на больших датасетах без попарного сравнения.

Шинглы — 3-граммы на нормализованном тексте.
Порог Jaccard similarity — 0.8 (сообщения считаются дубликатами
при 80% совпадении шинглов).

In [9]:
_NUM_PERM = 128
_JACCARD_THRESHOLD = 0.8
_SHINGLE_SIZE = 3

In [10]:
def _shingles(text: str, k: int = _SHINGLE_SIZE) -> set:
    """Разбивает текст на k-шинглы (символьные n-граммы)."""
    if len(text) < k:
        return {text}
    return {text[i:i+k] for i in range(len(text) - k + 1)}

In [11]:
def _make_minhash(text: str, num_perm: int = _NUM_PERM) -> MinHash:
    """Создаёт MinHash сигнатуру для текста."""
    mh = MinHash(num_perm=num_perm)
    for s in _shingles(text):
        mh.update(s.encode('utf-8'))
    return mh

Нормализуем текст для MinHash

In [12]:
_norm_texts = df_4['text'].apply(_normalize_for_dedup).tolist()

Создаём LSH индекс

In [13]:
lsh = MinHashLSH(threshold=_JACCARD_THRESHOLD, num_perm=_NUM_PERM)
_minhashes = []
for _i, _text in enumerate(_norm_texts):
    _mh = _make_minhash(_text)
    _minhashes.append(_mh)
    lsh.insert(_i, _mh)

Находим кластеры дубликатов

In [14]:
_visited = set()
_clusters = []
for _i in range(len(_minhashes)):
    if _i in _visited:
        continue
    _result = lsh.query(_minhashes[_i])
    if len(_result) > 1:
        _clusters.append(_result)
        _visited.update(_result)
    else:
        _visited.add(_i)

Оставляем первый элемент каждого кластера, остальные удаляем

In [15]:
_to_remove = set()
for _cluster in _clusters:
    for _idx in _cluster[1:]:
        _to_remove.add(_idx)

df_5 = df_4.drop(df_4.index[list(_to_remove)]).reset_index(drop=True)

In [16]:
print(f'MinHash LSH: найдено {len(_clusters)} кластеров near-дубликатов')
print(f'Удалено через MinHash LSH: {len(_to_remove)} строк')
print(f'После MinHash LSH: {len(df_5)} строк')

MinHash LSH: найдено 4505 кластеров near-дубликатов
Удалено через MinHash LSH: 7782 строк
После MinHash LSH: 98808 строк


Примеры найденных MinHash LSH кластеров

In [17]:
print(f'Найдено кластеров near-дубликатов: {len(_clusters)}')
for _ci, _cluster in enumerate(_clusters[:5]):
    print(f'\n--- Кластер {_ci+1} (размер {len(_cluster)}) ---')
    for _idx in _cluster[:3]:
        _text = str(df_4.iloc[_idx]['text'])[:120]
        _label = df_4.iloc[_idx]['label']
        print(f'  [label={_label}] {_text}')
    if len(_cluster) > 3:
        print(f'  ... и ещё {len(_cluster) - 3} сообщений')

Найдено кластеров near-дубликатов: 4505

--- Кластер 1 (размер 7) ---
  [label=0] Спасибо большое за отзыв🥰
  [label=0] Увидела, спасибо большое за ответ🙏❤️
  [label=0] Спасибо большое за ответ!
  ... и ещё 4 сообщений

--- Кластер 2 (размер 6) ---
  [label=0] Пасибо.
  [label=0] Спасибо
  [label=0] Cпасибо
  ... и ещё 3 сообщений

--- Кластер 3 (размер 2) ---
  [label=0] а когда закончится обработка?
  [label=0] Когда закончится обработка?

--- Кластер 4 (размер 5) ---
  [label=0] Урааа
  [label=0] ураааааа!!!!!!!!!!
  [label=0] Урааааа🙈👍😂
  ... и ещё 2 сообщений

--- Кластер 5 (размер 3) ---
  [label=0] Понял , спасибо
  [label=0] а, понял, спасибо
  [label=0] Понял
Спасибо!


#### Удаление пересечений с тестовой выборкой (Data Leakage)

Проверяется пересечение обучающих данных с:
- `test_clear.json` — тестовая выборка чистых сообщений
- `parsed_lols_2023.txt` — тестовая выборка спам-сообщений

Найденные пересечения удаляются, чтобы избежать утечки данных
из тестовой выборки в обучающую.

In [18]:
if os.path.exists(RAW_DIR / 'test_clear.json'):
    df_test_clear = pd.read_json(RAW_DIR / 'test_clear.json')
    test_texts = set(df_test_clear['text'].tolist())
    _before_leak = len(df_5)
    df_6 = df_5[~df_5['text'].isin(test_texts)]
    _removed = _before_leak - len(df_6)
    if _removed > 0:
        print(f'Удалено {_removed} текстов, пересекающихся с тестовой выборкой (data leakage)')
    print(f'После очистки: {len(df_6)} строк')
else:
    df_6 = df_5

Удалено 21081 текстов, пересекающихся с тестовой выборкой (data leakage)
После очистки: 77727 строк


In [19]:
if os.path.exists(EXTERNAL_DIR / 'parsed_lols_2023.txt'):
    lines_2023 = load_txt_lines(EXTERNAL_DIR / 'parsed_lols_2023.txt')
    test_spam_texts = set((l.replace('<br/>', '', 1).strip().replace('<br/>', '\n') for l in lines_2023 if l.strip()))
    _before_leak = len(df_6)
    df_7 = df_6[~df_6['text'].isin(test_spam_texts)]
    _removed = _before_leak - len(df_7)
    if _removed > 0:
        print(f'Удалено {_removed} текстов, пересекающихся с parsed_lols_2023 (data leakage)')
    print(f'После очистки: {len(df_7)} строк')
else:
    df_7 = df_6

Удалено 201 текстов, пересекающихся с parsed_lols_2023 (data leakage)
После очистки: 77526 строк


#### Итоги очистки

Выводится финальная статистика по количеству строк и соотношению классов.

In [20]:
df_8 = df_7.reset_index(drop=True)

In [21]:
print(f'Итого после очистки: {len(df_8)} строк')
print(f"  Ham (0): {len(df_8[df_8['label'] == 0])}")
print(f"  Spam (1): {len(df_8[df_8['label'] == 1])}")
print(f"  Соотношение: {len(df_8[df_8['label'] == 1]) / len(df_8[df_8['label'] == 0]):.2f}")

Итого после очистки: 77526 строк
  Ham (0): 59086
  Spam (1): 18440
  Соотношение: 0.31


## Сохранение очищенного датасета

Результат сохраняется в `data/interim/cleaned.csv` для использования в следующем ноутбуке.

In [22]:
df_8.to_csv(INTERIM_DIR / 'cleaned.csv', index=False)
print(f'Сохранено: {len(df_8)} строк в {INTERIM_DIR / "cleaned.csv"}')

Сохранено: 77526 строк в /home/sophrosyne/STANKIN_AntiSpam_Bot/data/interim/cleaned.csv
